# Project: European Air Traffic Pipeline on EUROCONTROL OPDI data (Medallion Architecture, PySpark)

## DAT535 - Data Engineering

### Thomas Canisares--Morère & Mathis Nancey - Group 08

Batch pipeline that turns EUROCONTROL OPDI flight data into clean, analysis-ready datasets using the Medallion Architecture.

- Period: June 1, 2026 -> July 28, 2026 (~5.5 GB)
- Tables: flight_list (monthly), flight_events and measurements (10-day intervals), OurAirports repositories (airports, runways)

This notebook walks through the pipeline layer by layer and shows the evidence of each one. The code lives in the
`opdi_pipeline` package (`projet_opdi/src/opdi_pipeline/`); the same code runs in GitHub Actions
(`python projet_opdi/run_pipeline.py --env <dev|prod>`). By default the notebook reads the reports of the last run
instead of running the pipeline again.

In [1]:
import json
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / "src"))      # projet_opdi/src
import opdi_pipeline                                     # first: on the VM, findspark points to Spark 3.5 (/opt/spark)
from opdi_pipeline.config import load_settings
from opdi_pipeline.spark import get_spark

ENV = "prod"                     # "prod": full scope (~/opdi-data) | "dev": 10-day sample (~/opdi-data-dev)
settings = load_settings(ENV)
spark = get_spark(settings)
print(f"{ENV}: scope [{settings.start}, {settings.end}[ | data {settings.data_root} | Spark {spark.version}")

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


26/11/09 19:37:44 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


prod: scope [2026-06-01, 2026-07-29[ | data /home/ubuntu/opdi-data | Spark 3.5.0


In [2]:
def show(rows, columns=None):
    """Display a list of dicts as a table (pandas if available, plain text otherwise)."""
    try:
        import pandas as pd
        from IPython.display import display
        display(pd.DataFrame(rows, columns=columns))
    except ImportError:
        for r in rows:
            print(r)

In [3]:
RUN_PIPELINE = False     # True: run the layers now (long on prod) | False: show the results of the last run

from opdi_pipeline import bronze, gold, ingestion, quality, silver

def latest_run_id(settings):
    """Run id of the most recent complete run (the one with a summary report)."""
    runs = sorted((settings.data_root / "reports").glob("*/summary.json"), key=lambda p: p.stat().st_mtime)
    return runs[-1].parent.name if runs else None

if RUN_PIPELINE:
    reports = {"ingest": ingestion.run(settings), "bronze": bronze.run(spark, settings),
               "silver": silver.run(spark, settings), "gold": gold.run(spark, settings)}
    reports["summary"] = quality.run(spark, settings)
else:
    run_dir = settings.data_root / "reports" / latest_run_id(settings)
    reports = {p.stem: json.loads(p.read_text()) for p in run_dir.glob("*.json")}
print("Run", reports["summary"]["run_id"], "| status", reports["summary"]["status"],
      "| generated", reports["summary"]["generated_at_utc"])

Run prod-8-1 | status PASSED | generated 2026-11-09T19:00:53+00:00


## Part 1: Ingestion (landing zone)

The main loop that downloads the files and records the results in a manifest CSV file.
The manifest contains all the relevant information about each file downloaded and it contains the following columns:
- run_id: a unique identifier for this run of the script
- table: the table name (flight_list, flight_events, measurements, reference)
- file: the file name
- url: the URL from which the file was downloaded
- http_status: the HTTP status code returned by the server
- size_bytes: the size of the file in bytes
- last_modified: the last modified date of the file on the server (if available)
- result: the result of the download (downloaded, already_present, absent, empty, failed)
- sha256: the SHA256 hash of the file
- local_path: the local path of the downloaded file
- retrieved_at_utc: the UTC time when the file was retrieved

Code: `opdi_pipeline/ingestion.py`.

In [4]:
r = reports["ingest"]
print("Files of the scope:", r["files_expected"], "| results:", r["results"], "| sizes (GB):", r["size_gb"])
print("Problems:", r["problems"] or "none")
# "republished": the source published a new version of a file we had (other size or Last-Modified) -> downloaded again

Files of the scope: 16 | results: {'already_present': 16} | sizes (GB): {'flight_list': 0.1, 'flight_events': 3.034, 'measurements': 2.387, 'reference': 0.017}
Problems: none


## Part 2: Bronze layer
### Schema validation

OPDI files from different periods were observed with different schemas (e.g. `long` vs `decimal(20,0)`
identifiers, dates stored as `string` vs `timestamp`). Before loading anything into Silver, we check that
all the files of a same table share the same schema, and we record each file's schema fingerprint and
row count in `schema_report.csv` for traceability.

### Bronze tables (row-level lineage)

The source files in `raw/` are kept untouched. The Bronze tables in `bronze/<table>/` contain
exactly the same rows, columns and types, plus row-level lineage columns taken from the manifest:

| Column | Meaning |
|---|---|
| `_source_file` | name of the raw file the row comes from |
| `_source_period` | month (`YYYYMM`) or 10-day window start (`YYYYMMDD`) of the source file — also the partition key |
| `_ingestion_run_id` / `_ingested_at_utc` | download run that retrieved the file, and when |
| `_source_last_modified` | publication date of the file on the EUROCONTROL server |
| `_source_sha256` | fingerprint of the source file |
| `_bronze_run_id` / `_bronze_loaded_at_utc` | run that built this Bronze period, and when |

No cleaning or type conversion happens here. Tables are partitioned by source period, so re-running only
replaces the partitions of the current period. A reconciliation then checks that every raw row is in Bronze.

### Incremental build

`bronze/_state.json` records the SHA-256 of the source file behind every Bronze period. A run only rewrites the
periods whose file is new or was republished (dynamic partition overwrite); the reconciliation still checks every
file of the scope. `--full-refresh` rebuilds everything, e.g. after a change of the Bronze code.

Code: `opdi_pipeline/bronze.py`.

In [5]:
r = reports["bronze"]
print("Schema variants per table:", r["schema_variants"], "| files off the schema contract:",
      r.get("files_off_contract") or "none")
print(f"Reconciliation: {'PASSED' if r['reconciliation_passed'] else 'FAILED'} | files {r['files_reconciled']}/"
      f"{r['files_expected']} | rows {r['rows']:,} | size {r['size_gb']} GB")
built = r.get("files_built", [])
print(f"Incremental build: {len(built)} file(s) (re)built, {r.get('files_up_to_date', 0)} already up to date"
      f"{' (full refresh)' if r.get('full_refresh') else ''}:", built or "none")
show(r["reconciliation"])

Schema variants per table: {'flight_list': 1, 'flight_events': 1, 'measurements': 1} | files off the schema contract: none
Reconciliation: PASSED | files 16/16 | rows 208,888,834 | size 5.46 GB
Incremental build: 0 file(s) (re)built, 16 already up to date: none


,table,file,raw_rows,bronze_rows,missing_lineage,status
0,flight_list,flight_list_202606.parquet,1558373,1558373,0,OK
1,flight_list,flight_list_202607.parquet,1746344,1746344,0,OK
2,flight_events,flight_events_20260629_20260709.parquet,14899719,14899719,0,OK
3,flight_events,flight_events_20260719_20260729.parquet,15640487,15640487,0,OK
4,flight_events,flight_events_20260619_20260629.parquet,14312202,14312202,0,OK
5,flight_events,flight_events_20260709_20260719.parquet,15543261,15543261,0,OK
6,flight_events,flight_events_20260530_20260609.parquet,13191492,13191492,0,OK
7,flight_events,flight_events_20260609_20260619.parquet,13082972,13082972,0,OK
8,measurements,measurements_20260719_20260729.parquet,21820246,21820246,0,OK
9,measurements,measurements_20260530_20260609.parquet,17608700,17608700,0,OK


### Bronze summary

**Scope:** June 1, 2026 -> July 28, 2026 (EUROCONTROL OPDI v0.0.2 + OurAirports reference data).

| Table | Source files | Partitions (`_source_period`) | Rows | Raw size |
|---|---|---|---|---|
| flight_list | 2 monthly files | 2 | 3,304,717 | 0.10 GB |
| flight_events | 6 ten-day windows | 6 | 86,670,133 | 3.03 GB |
| measurements | 6 ten-day windows | 6 | 118,779,222 | 2.39 GB |
| airports | 1 CSV | – | 86,258 | 0.02 GB (both CSVs) |
| runways | 1 CSV | – | 48,345 | |
| **Total** | **16 files** | | **208,888,675** | **~5.5 GB** |

**Key Bronze layer properties**
1. **Schema preservation:** original columns and types are kept unchanged (no cleaning, no casting).
2. **Row-level lineage:** every record carries `_source_file`, `_source_period`, `_ingestion_run_id`, `_ingested_at_utc`,
   `_source_last_modified`, `_source_sha256`, `_bronze_run_id` and `_bronze_loaded_at_utc`.
3. **Reconciliation: PASSED.** 16/16 files have the same row count in `raw/` and `bronze/`, and no row is missing its lineage.
4. **Partitioning:** by `_source_period`, with dynamic partition overwrite, so a re-run only replaces the periods it writes.
5. **Storage:** Bronze takes 5.46 GB, slightly *less* than the raw files. Lineage values are constant within a file,
   so Parquet dictionary encoding stores them almost for free.

**Findings that drive the Silver layer**
- One schema variant per table for the whole period: no harmonisation needed (older OPDI periods, e.g. 2025,
  use `decimal` ids, `timestamp_ntz` dates and an extra `unix_time` column and would need it).
- Dates and times (`first_seen`, `last_seen`, `event_time`) are **strings** and must be cast in Silver.
- **Period boundaries:** the first event window starts on May 30 (before the scope) and the last one ends on July 28,
  while `flight_list_202607` covers all of July. Silver must filter on the scope dates.
- `info` (flight_events) is a JSON string; `adep`/`ades` are often missing (NULL or `""`); callsigns can be padded;
  some altitudes are negative.

**Known limitation:** a file already present in `raw/` is not downloaded again. If EUROCONTROL republishes a file
(this happened for 4 event files in October 2026), the change is not detected. Comparing the server `Last-Modified`
with the manifest would fix this.

## Part 3: Silver layer

The profiling of the Bronze tables is in `01_data_understanding.ipynb`.

### Data model

The Silver layer turns the Bronze tables into typed, cleaned and linked tables. **Grain: one row per flight**,
with its events as a child table and two reference dimensions.

| Silver table | Grain | Key | Main columns |
|---|---|---|---|
| `silver/flights` | one flight | `flight_id` (long) | `callsign`, `transponder_icao24`, `registration`, `aircraft_typecode`, `aircraft_model`, `aircraft_class`, `operator_icao`, `origin_icao`, `dest_icao`, `route`, `flight_category`, origin/destination airport attributes, `distance_km` (great circle), `first_seen_utc`, `last_seen_utc` (timestamp), `flight_date`, `duration_min`, flags `has_adep`, `has_ades`, `is_short_flight` |
| `silver/flight_events` | one event of a flight | `event_id` (long), FK `flight_id` | `event_type`, `event_timestamp_utc`, `event_date`, `latitude`, `longitude`, `altitude_ft`, flag `is_altitude_valid`, `source`, `info` (JSON parsed as `map<string,string>`) |
| `silver/airports` | one airport with a 4-character ident (ICAO code when it exists) | `airport_icao` | name, type, country, continent, municipality, coordinates, elevation, IATA code |
| `silver/runways` | one runway of a known airport | `airport_icao` + `le_ident`/`he_ident` | length, width, surface, closed flag, headings |
| `quarantine/flights`, `quarantine/flight_events` | one rejected row | as above | all columns + `_quarantine_reason`, `_silver_flagged_at_utc` |

Every Silver row keeps its Bronze lineage (`_source_file`, `_source_period`, `_ingestion_run_id`, `_source_sha256`).

**Event types** (30 types observed in 3.1): flight milestones `first_seen`, `last_seen`, `take-off`, `landing`,
`top-of-climb`, `top-of-descent`, `level-start`, `level-end`, `first-xing-fl50/70/100/245`, `last-xing-fl50/70/100/245`;
airport events `entry-` / `exit-` `runway`, `taxiway`, `parking_position`, `apron`, `threshold`, `hangar`, `deicing_pad`.
The `info` JSON of the airport events holds OpenStreetMap data: `osm_airport` (ICAO code), `osm_aeroway`,
`osm_ref` (e.g. runway `"07-25"`), `osm_id`, `opdi_time_in_use_s`, `osn_flight_id`, `osn_icao24`.
Altitudes are in feet (mean altitude of the `first-xing-fl100` events ≈ 9,900 ft).

**Cleaning rules**
- **Reject (quarantine, with a reason):** missing flight id or transponder address, unparsable timestamps,
  `first_seen` > `last_seen`, flight longer than 24 h; events with no flight id, no type, unparsable time,
  coordinates out of range, or whose flight is not a valid Silver flight (`ORPHAN_EVENT`).
- **Flag and keep:** unknown departure/arrival airport (`has_adep`, `has_ades`; about half of the flights, still
  valid flights), flights shorter than 2 minutes (`is_short_flight`), implausible altitudes (`is_altitude_valid`).
- **Correct:** `""`, `"nan"` and NULL are turned into a single NULL; strings are trimmed; codes are upper-cased;
  airport codes that are not 4 characters long are set to NULL.
- **Out of scope (dropped, counted):** flights that started and events that happened outside [June 1, July 29[.
- **Duplicates:** one row per `flight_id` (the most complete one); one event per (`flight_id`, `event_type`, time).

**Not in Silver (intentionally):** the `measurements` table (cumulative distance flown and time passed since
`first_seen`, per event) is not needed for the Gold question (airport operations) and stays in Bronze; it is the
basis of a future flight-efficiency indicator. `adep_p`/`ades_p`, `dof` and `version` are dropped.

**Partitioning:** flights by `flight_year`/`flight_month`; events and quarantine tables by `_source_period`
(10-day window or month), the same layout as Bronze.

### Standardisation, quality rules, deduplication, enrichment

One explicit target schema per table: snake_case names, timestamps cast from strings (UTC), upper-case codes,
a single NULL for missing values, and the `info` JSON parsed into a map. The scope filter (June 1 – July 28) is
prepared here as a column, so that out-of-scope rows can be counted in 3.8.

Rejected rows go to quarantine with the first rule they fail; incomplete but valid rows are kept with a flag.

Flights are joined with the OurAirports dimension (departure and arrival), which gives the route, the flight
category (domestic / international) and the great-circle distance.

Silver is rebuilt as a whole for the project scope (`partitionOverwriteMode=static`: the previous version of each
table is replaced, including partitions that no longer exist). The ~86 M events are processed here, in two passes
only (valid events, then rejected events), instead of once per `count()`.

Code: `opdi_pipeline/silver.py` (one function per step, unit-tested in `tests/test_silver_rules.py`).

### Before-and-after evidence of quality

All figures are computed from the written tables, so they describe exactly what was published.
Row accounting: **Bronze rows = out of scope + quarantined + duplicates removed + Silver rows.**

In [6]:
r = reports["silver"]
acc = r["row_accounting"]
show([{"": k, "flights": acc["flights"][k], "flight_events": acc["events"][k]}
      for k in ("bronze", "out_of_scope", "quarantined", "duplicates", "silver")])
print("Accounting consistent:", r["accounting_consistent"], "| orphan events in Silver:", r["orphan_events"])
print("Flight outcomes:", r["flight_outcomes"])
print("Quarantine reasons:", r["quarantine_reasons"])

,,flights,flight_events
0,bronze,3304717,86670133
1,out_of_scope,175452,2660795
2,quarantined,625,9100
3,duplicates,810,0
4,silver,3127830,84000238


Accounting consistent: True | orphan events in Silver: 0
Flight outcomes: {'DURATION_GT_24H': 625, 'VALID': 3128640}
Quarantine reasons: {'flights': {'DURATION_GT_24H': 625}, 'flight_events': {'ORPHAN_EVENT': 9100}}


In [7]:
show(r["missing_values"])      # three forms of "missing" in Bronze -> a single NULL in Silver
print("Completeness:", r["completeness"])
print("Flight categories:", r["flight_categories"])
print("Altitude valid (%):", r["pct_altitude_valid"])
print("Types Bronze -> Silver:", r["types"])
show([{"event_type": k, "events": v} for k, v in r["event_types"].items()])

,bronze_column,bronze_pct_null,bronze_pct_empty,bronze_pct_nan,silver_column,silver_pct_null
0,flt_id,8.31,3.64,0.0,callsign,11.88
1,icao_operator,55.65,0.00,0.0,operator_icao,55.58
2,typecode,20.78,0.00,0.0,aircraft_typecode,20.74
3,adep,43.50,0.00,0.0,origin_icao,43.49
4,ades,40.31,0.00,0.0,dest_icao,40.24


Completeness: {'pct_both_airports_known': 38.51, 'pct_short_flights': 2.3, 'pct_origin_found_in_ourairports': 56.44}
Flight categories: {'UNKNOWN': 1925418, 'INTERNATIONAL': 872829, 'DOMESTIC': 329583}
Altitude valid (%): 99.95
Types Bronze -> Silver: [{'bronze': 'first_seen string', 'silver': 'first_seen_utc timestamp'}, {'bronze': 'event_time string', 'silver': 'event_timestamp_utc timestamp'}, {'bronze': 'info string', 'silver': 'info map<string,string>'}]


,event_type,events
0,level-start,31046197
1,level-end,20974519
2,first_seen,3127830
3,last_seen,3126972
4,first-xing-fl50,1989765
5,last-xing-fl50,1989557
6,first-xing-fl70,1943240
7,last-xing-fl70,1943039
8,entry-runway,1910031
9,exit-runway,1909435


## Part 4: Gold Layer
### Construction

**Question served:** *how busy is each European airport, how are its runways used and how long do ground phases
take, by day and by hour?* (showcase airports: Toulouse-Blagnac LFBO and Stavanger ENZV).

Gold tables are small, denormalised and aggregated, so a dashboard can query them without joins.

| Gold table | Grain | Content |
|---|---|---|
| `gold/airport_daily_ops` | airport × day | departures, arrivals, peak hour, operators, aircraft types, shares |
| `gold/airport_hourly_ops` | airport × day × hour (UTC) | departures, arrivals, movements |
| `gold/runway_usage` | airport × runway × day | movements per runway + runway characteristics (OurAirports) |
| `gold/airport_ground_times` | airport × day | taxi-out / taxi-in (mean, median, p90) + coverage |

**Airport milestones of a flight.** Airport events carry the airport in their `info` JSON (`osm_airport`, ICAO code)
and the airport element in `osm_ref` (e.g. runway `"07-25"`). Each runway / parking event is therefore attached to
the departure or to the arrival airport of the flight:
- departure time = last `entry-runway` at the departure airport (line-up); arrival time = first `exit-runway` at the
  arrival airport (runway vacated). Fallback: `take-off` / `landing` events, then `first_seen` / `last_seen`
  (`time_source` and `pct_time_from_event` say how often the precise time was available);
- taxi-out = `exit-parking_position` -> departure runway entry; taxi-in = arrival runway exit -> `entry-parking_position`
  (kept between 1 and 120 minutes). Parking positions are only mapped at some airports: see the coverage columns.

All times are **UTC** (not airport local time).

**Gold pipeline structure**
- **4.1 Construction:** the four tables above.
- **4.2 Validation:** reconciliation with Silver, consistency and range checks, coverage figures, showcase airports.
- **4.3 Publication:** `_SUCCESS` marker and run record in `gold/run_status`, only if 4.2 passes.

Code: `opdi_pipeline/gold.py`.

### The Gold queries (Spark SQL)

The Gold tables are built with Spark SQL on temporary views of the Silver tables (`silver_flights`, `silver_events`, `silver_runways`, `airports_dim`): one query per step in `opdi_pipeline/sql/`. Silver uses the DataFrame API (step-by-step rules), Gold uses SQL (declarative aggregations for a consumer).

In [8]:
for name in ["flights_timed", "airport_movements", "airport_daily_ops"]:
    print(f"-- ===== {name}.sql =====")
    print(gold.sql_text(name))

-- ===== flights_timed.sql =====
-- Airport milestones of each flight, from its take-off / landing / runway / parking events.
-- Airport events give their airport in info['osm_airport'] (ICAO code) and the airport element in
-- info['osm_ref'] (e.g. runway "07-25"): each event is attached to the departure or to the arrival airport.
WITH ground_events AS (
    SELECT e.flight_id,
           e.event_type,
           e.event_timestamp_utc  AS t,
           e.info['osm_airport']  AS event_airport,
           e.info['osm_ref']      AS element_ref,
           f.origin_icao,
           f.dest_icao
    FROM silver_events e
    JOIN silver_flights f ON e.flight_id = f.flight_id
    WHERE e.event_type IN ('take-off', 'landing', 'entry-runway', 'exit-runway',
                           'exit-parking_position', 'entry-parking_position')
),
milestones AS (
    SELECT flight_id,
           MIN(CASE WHEN event_type = 'take-off' THEN t END) AS takeoff_utc,
           MAX(CASE WHEN event_type = 'landin

### Validation and publication

Gold is published (`_SUCCESS` marker + record in `gold/run_status`) only if every check passes: no empty table, no movement lost or duplicated by the aggregations, reconciliation with the Silver flights, ranges and consistency of the counts, percentages and durations, and **schema contracts** (`opdi_pipeline/schemas.py`): every published table must keep the columns and types its consumers expect.

In [9]:
r = reports["gold"]
show([{"check": k, "passed": v} for k, v in r["checks"].items()])
print("Movements by source of the time:", r["movements_by_time_source"])
print("Coverage (%):", r["coverage"])
print("Showcase airports:", json.dumps(r["showcase"], indent=1, default=str))

,check,passed
0,airport_hourly_ops is not empty,True
1,airport_daily_ops is not empty,True
2,runway_usage is not empty,True
3,airport_ground_times is not empty,True
4,arrival_reference is not empty,True
5,airport_arrival_hourly is not empty,True
6,airport_arrival_daily is not empty,True
7,holding_hotspots is not empty,True
8,daily movements = airport_movements,True
9,hourly movements = airport_movements,True


Movements by source of the time: {'DEP / runway event': 859227, 'DEP / take-off / landing event': 24826, 'DEP / first / last seen': 883555, 'ARR / runway event': 843036, 'ARR / first / last seen': 977682, 'ARR / take-off / landing event': 47982}
Coverage (%): {'pct_movements_timed_by_event': 48.8, 'pct_movements_with_runway': 43.3, 'pct_movements_with_taxi_time': 2.5, 'pct_runway_movements_matched_ourairports': 99.2}
Showcase airports: {
 "LFBO": {
  "days": 58,
  "movements": 12070,
  "per_day": 208.1,
  "pct_jet": 87.4,
  "pct_time_from_event": 90.0
 },
 "ENZV": {
  "days": 58,
  "movements": 8617,
  "per_day": 148.6,
  "pct_jet": 53.1,
  "pct_time_from_event": 10.6
 }
}


In [10]:
from pyspark.sql.functions import col
daily = spark.read.parquet(str(settings.gold_dir / "airport_daily_ops"))
(daily.filter(col("airport_icao") == "LFBO")
      .select("movement_date", "departures", "arrivals", "movements", "peak_hour_movements",
              "distinct_operators", "pct_jet", "pct_international", "pct_time_from_event")
      .orderBy("movement_date").show(10))

+-------------+----------+--------+---------+-------------------+------------------+-------+-----------------+-------------------+
|movement_date|departures|arrivals|movements|peak_hour_movements|distinct_operators|pct_jet|pct_international|pct_time_from_event|
+-------------+----------+--------+---------+-------------------+------------------+-------+-----------------+-------------------+
|   2026-06-01|       131|     131|      262|                 20|                30|   89.5|             47.1|               86.6|
|   2026-06-02|       107|     116|      223|                 18|                25|   84.0|             45.9|               87.0|
|   2026-06-03|       113|     121|      234|                 20|                24|   86.1|             40.4|               89.7|
|   2026-06-04|       119|     130|      249|                 21|                27|   89.3|             42.2|               90.4|
|   2026-06-05|       118|     135|      253|                 23|                27

### Arrival efficiency indicators (ASMA-like & Descent Profiles)

The Silver arrival tables (, ) compute the 100 NM ring entry, transit duration, vertical level-offs, and holding patterns. The Gold arrival tables aggregate additional transit time, level minutes, and estimated excess fuel / CO2.


In [11]:
arr = spark.read.parquet(str(settings.silver_dir / "arrivals"))
(arr.filter(col("dest_icao") == "LFBO")
    .select("flight_id", "origin_icao", "entry_sector", "arrival_runway", "transit_min", "level_minutes", "holding_count")
    .show(5))

arr_daily = spark.read.parquet(str(settings.gold_dir / "airport_arrival_daily"))
(arr_daily.filter(col("airport_icao") == "LFBO")
          .select("movement_date", "arrivals", "arrivals_measured", "avg_additional_min", "avg_level_minutes", "pct_with_holding", "est_fuel_kg")
          .show(5))


+--------------------+-----------+------------+--------------+-----------+-------------+-------------+
|           flight_id|origin_icao|entry_sector|arrival_runway|transit_min|level_minutes|holding_count|
+--------------------+-----------+------------+--------------+-----------+-------------+-------------+
|-8867531300601981809|       LFCC|        NULL|          NULL|       NULL|          0.5|            0|
|-8254090745176380361|       EGLL|           N|       14R-32L|      24.95|         0.83|            0|
|-8132795365890759360|       LFPO|           N|       14R-32L|      20.98|          0.0|            0|
|-8127124889376651548|       EDHI|           N|       14R-32L|       25.4|         2.17|            0|
|-7490843737112175879|       EBCI|           N|       14R-32L|      23.38|         1.08|            0|
+--------------------+-----------+------------+--------------+-----------+-------------+-------------+
only showing top 5 rows



+-------------+--------+-----------------+------------------+-----------------+----------------+-----------+
|movement_date|arrivals|arrivals_measured|avg_additional_min|avg_level_minutes|pct_with_holding|est_fuel_kg|
+-------------+--------+-----------------+------------------+-----------------+----------------+-----------+
|   2026-06-17|     133|               96|              1.77|             1.18|             0.8|     1997.7|
|   2026-06-21|      97|               53|              1.22|             1.69|             1.0|     2077.5|
|   2026-06-24|     123|               83|              0.57|             1.57|             0.0|     2450.3|
|   2026-06-27|      76|               48|              1.26|             1.81|             0.0|     1745.2|
|   2026-07-05|      88|               79|              0.78|             1.97|             1.1|     2206.5|
+-------------+--------+-----------------+------------------+-----------------+----------------+-----------+
only showing top 5 

## Part 5: Validation summary

This section gathers the results of the checks made in each layer (Bronze reconciliation, Silver row accounting and
referential integrity, Gold validation and publication), the volumes and storage of every table, and the quarantine
reasons. It is written to `pipeline_summary.json` and shown in the GitHub Actions run summary; the run fails if one quality gate fails.

In [12]:
s = reports["summary"]
show([{"gate": k, "passed": v} for k, v in s["gates"].items()])
show(s["tables"])
print("Slowest steps (s):", sorted(s["step_durations_s"].items(), key=lambda kv: kv[1], reverse=True)[:8])

,gate,passed
0,Bronze: raw files match the schema contract,True
1,Bronze: raw -> Bronze reconciliation,True
2,Silver: row accounting consistent,True
3,Silver: no orphan event,True
4,Silver: tables match the schema contract,True
5,Gold: validation checks,True
6,Gold: published (_SUCCESS marker),True


,layer,table,rows,size_gb
0,Bronze,flight_list,3304717,0.1054
1,Bronze,flight_events,86670133,3.0578
2,Bronze,measurements,118779222,2.2895
3,Bronze,airports,86350,0.0056
4,Bronze,runways,48412,0.0019
5,Silver,flights,3127830,0.1741
6,Silver,flight_events,84000238,2.5112
7,Silver,airports,39316,0.0018
8,Silver,runways,37408,0.0005
9,Silver,arrivals,1866643,0.1018


Slowest steps (s): [('silver: write silver/flight_events', 263.9), ('silver: arrivals', 215.7), ('silver: arrivals: write silver/descent_level_offs', 153.3), ('silver: write quarantine/flight_events', 104.7), ('silver: arrivals: write silver/arrivals', 55.8), ('ingest: download', 35.7), ('ml: train GBTRegressor & baseline', 33.3), ('gold: airport movements', 28.5)]


## Part 6: Conclusion and limits


### 6.1 What the pipeline delivers

1. **End-to-end Medallion pipeline** (Bronze -> Silver -> Gold) in PySpark on EUROCONTROL OPDI data for June 1 – July 28,
   2026: 208.9 M raw rows (~5.5 GB) from 16 source files, reconciled row by row between `raw/` and Bronze.
2. **Traceability:** a download manifest (URL, `Last-Modified`, SHA-256, run id) and row-level lineage columns in every
   Bronze and Silver row, so any Gold figure can be traced back to its source file.
3. **Data quality:** explicit reject rules with a quarantine reason, flag-and-keep rules for incomplete but valid
   records (e.g. unknown departure or arrival airport), a single representation of missing values, and
   before/after evidence computed from the published tables (3.8).
4. **Gold tables for airport operations:** daily and hourly traffic, runway usage and taxi-out / taxi-in times for
   every airport, with coverage columns that say how often the precise information was available.
5. **Operation:** the same notebook runs interactively on the VM and headlessly in GitHub Actions
   (dev: one 10-day window, prod: full scope after approval, monthly schedule), with unit tests and an end-to-end
   test on synthetic data.

*Key figures of the prod run (quarantine rates, coverage, measured speed-ups of 3.9) are to be added here from the
outputs of Parts 3.8, 3.9, 4.2 and 5.*

---

### 6.2 Limits

1. **ADS-B coverage:** OPDI is built from OpenSky Network receivers, whose density varies across Europe. Ground
   events (parking positions, taxiways) only exist where receivers see aircraft on the ground, so taxi times and
   runway usage cover part of the flights only (see the coverage columns).
2. **Unknown airports:** about half of the flights have no detected departure or arrival airport. They stay in Silver
   (flagged) but cannot contribute to airport indicators.
3. **Fixed period and storage:** two months only, because of the VM disk (Bronze + Silver + Gold must fit next to the
   raw files). Older OPDI periods use other schemas (e.g. `decimal` ids) and would need a harmonisation step.
4. **Republished source files** are not re-downloaded (a file already in `raw/` is kept). Comparing the server
   `Last-Modified` with the manifest would detect them.
5. **UTC times:** hourly indicators are in UTC, not in the airport's local time.
6. **Runway references** come from OpenStreetMap (`info` JSON) and are matched to OurAirports runway ends by name;
   unmatched references keep empty runway characteristics.

---

### 6.3 Future work

- Incremental runs per period (only new or republished files), and a longer history on external storage.
- Flight-efficiency indicators from the `measurements` table (distance flown vs great-circle distance).
- Comparison of the Gold traffic with EUROCONTROL's published airport traffic dataset as an automated check.
- Local-time hourly indicators, using the airport time zone.